# Tareas comunes de NLP — Mejorando nuestro bot

En esta lección vamos a:
1. Aprender técnicas de NLP: tokenización, sentimiento, frases sustantivas
2. Usar TextBlob para analizar texto
3. Mejorar nuestro bot para que entienda sentimiento y contexto

## 1. Instalar TextBlob

TextBlob es una librería que simplifica el NLP. Tiene NLTK integrado.

In [ ]:
!pip install -U textblob
!python -m textblob.download_corpora

## 2. Conceptos clave de NLP

| Técnica | Qué hace | Ejemplo |
|---------|----------|--------|
| **Tokenización** | Dividir texto en palabras | "Hola mundo" → ["Hola", "mundo"] |
| **Sentimiento** | ¿Positivo o negativo? | "Me encanta" → positivo |
| **Frase sustantiva** | Identificar de qué se habla | "lovely cat" → "cat" |
| **Polaridad** | Qué tan positivo/negativo | -1.0 a 1.0 |
| **Subjetividad** | Qué tan objetivo/subjetivo | 0.0 a 1.0 |

## 3. Análisis de sentimiento con TextBlob

TextBlob analiza el sentimiento de un texto y devuelve polaridad (-1 a 1) y subjetividad (0 a 1).

In [ ]:
from textblob import TextBlob

# Ejemplos de sentimiento
textos = [
    "I love this, it's amazing!",
    "This is terrible, I hate it.",
    "The weather is ok.",
    "I am so happy today!",
    "This is the worst day ever."
]

for texto in textos:
    blob = TextBlob(texto)
    print(f"Texto: {texto}")
    print(f"  Polaridad: {blob.polarity:.2f} (-1=negativo, 1=positivo)")
    print(f"  Subjetividad: {blob.subjectivity:.2f} (0=objetivo, 1=subjetivo)")
    print()

### ¿Cómo interpretar?

| Polaridad | Significado |
|-----------|-------------|
| -1.0 a -0.5 | Muy negativo |
| -0.5 a 0 | Algo negativo |
| 0 | Neutral |
| 0 a 0.5 | Algo positivo |
| 0.5 a 1.0 | Muy positivo |

## 4. Extracción de frases sustantivas

TextBlob puede identificar **de qué estás hablando** — las frases sustantivas.

In [ ]:
from textblob import TextBlob
from textblob.np_extractors import ConllExtractor

# Crear extractor de frases sustantivas
extractor = ConllExtractor()

# Ejemplos
textos = [
    "I saw a lovely cat in the park",
    "The big brown fox jumped over the lazy dog",
    "I have a cool dog at home",
    "The weather in Paris is beautiful"
]

for texto in textos:
    blob = TextBlob(texto, np_extractor=extractor)
    print(f"Texto: {texto}")
    print(f"  Frases sustantivas: {list(blob.noun_phrases)}")
    print()

## 5. El bot mejorado

Ahora combinamos todo en un bot que:
1. Analiza el **sentimiento** de lo que decís
2. Responde acorde al sentimiento
3. Detecta **de qué estás hablando** (frase sustantiva)
4. Te pregunta más sobre ese tema

In [ ]:
from textblob import TextBlob
from textblob.np_extractors import ConllExtractor

extractor = ConllExtractor()

print("Hello, I am Marvin, the friendly robot.")
print("You can end this conversation at any time by typing 'bye'")
print("After typing each answer, press 'enter'")
print("How are you today?")

while True:
    user_input = input("> ")
    
    if user_input.lower() == "bye":
        break
    
    # Crear TextBlob con extractor de frases sustantivas
    user_input_blob = TextBlob(user_input, np_extractor=extractor)
    
    # Determinar respuesta según sentimiento
    if user_input_blob.polarity <= -0.5:
        response = "Oh dear, that sounds bad. "
    elif user_input_blob.polarity <= 0:
        response = "Hmm, that's not great. "
    elif user_input_blob.polarity <= 0.5:
        response = "Well, that sounds positive. "
    elif user_input_blob.polarity <= 1:
        response = "Wow, that sounds great. "
    
    # Si hay frase sustantiva, preguntar sobre ella
    if user_input_blob.noun_phrases:
        # Tomar la primera frase sustantiva y pluralizarla
        noun = user_input_blob.noun_phrases[0]
        response += f"Can you tell me more about {noun.pluralize()}?"
    else:
        response += "Can you tell me more?"
    
    print(response)

print("It was nice talking to you, goodbye!")

## 6. ¿Qué hace el código?

### Paso 1: Crear el extractor
```python
extractor = ConllExtractor()
```
Este extractor usa un modelo entrenado para encontrar frases sustantivas.

---

### Paso 2: Analizar sentimiento
```python
user_input_blob = TextBlob(user_input, np_extractor=extractor)
```
TextBlob analiza el texto y calcula polaridad (-1 a 1).

---

### Paso 3: Elegir respuesta según sentimiento
```python
if user_input_blob.polarity <= -0.5:
    response = "Oh dear, that sounds bad. "
elif user_input_blob.polarity <= 0:
    response = "Hmm, that's not great. "
...
```
4 gradientes de sentimiento: muy negativo, negativo, positivo, muy positivo.

---

### Paso 4: Detectar frase sustantiva
```python
if user_input_blob.noun_phrases:
    noun = user_input_blob.noun_phrases[0]
    response += f"Can you tell me more about {noun.pluralize()}?"
```
Si detecta "lovely cat", pregunta "Can you tell me more about lovely cats?"

## 7. Ejemplo de interacción

```
Hello, I am Marvin, the friendly robot.
How are you today?
> I am ok
Well, that sounds positive. Can you tell me more?
> I went for a walk and saw a lovely cat
Well, that sounds positive. Can you tell me more about lovely cats?
> cats are the best. But I also have a cool dog
Wow, that sounds great. Can you tell me more about cool dogs?
> I have an old hounddog but he is sick
Hmm, that's not great. Can you tell me more about old hounddogs?
> bye
It was nice talking to you, goodbye!
```

**¿Notaste la diferencia con el bot de la lección 1?**
- Ahora responde según el sentimiento
- Detecta de qué estás hablando
- Te pregunta más sobre ese tema

¡Es mucho más creíble!

## 8. Reflexión

### Preguntas para pensar:

1. **¿Las respuestas simpáticas engañarían a alguien?**
   - Más que el bot anterior, pero aún no es perfecto.

2. **¿Detectar la frase sustantiva hace al bot más creíble?**
   - Sí — parece que entiende de qué hablás.

3. **¿Por qué es útil extraer frases sustantivas?**
   - Para entender de qué trata el texto
   - Para clasificar documentos
   - Para hacer resúmenes

### ¿Qué falta para un bot realmente inteligente?

- **Memoria**: recordar oraciones anteriores
- **Contexto**: entender la conversación completa
- **Conocimiento**: tener información del mundo real
- **Razonamiento**: poder deducir cosas

## Resumen

1. **TextBlob** simplifica el NLP en Python
2. **Sentimiento**: analiza si un texto es positivo o negativo
3. **Frases sustantivas**: identifica de qué se habla
4. **Nuestro bot**: ahora responde según sentimiento y detecta temas

**Lección clave:** Combinando técnicas de NLP, podemos hacer bots más inteligentes y creíbles.